<a href="https://colab.research.google.com/github/noobylub/Computational-Linguistic/blob/master/SmallAttentionBasedLMClean_(with_TODOs).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A small decoder-only LM

In [ ]:
# All imports in one place

import math
import torch
from torch import nn
from torch.nn import CrossEntropyLoss
from torch.optim import AdamW
from torch.utils.data import TensorDataset, DataLoader

from tqdm.auto import tqdm

## Model code

In [ ]:
class LexicalEmbedding(nn.Module):
    """
    A simple wrapper around nn.Embedding, which takes special care to
    handle padding tokens.
    """
    def __init__(self, vocab_size, input_dim, padding_idx):
        super().__init__()
        self.emb = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=input_dim,
            padding_idx=padding_idx,
        )

    def forward(self, x: torch.Tensor):
        """
        This will accept both single and batched inputs.
        """
        return self.emb(x.long())  # (..., input_dim)


class Attention(nn.Module):
    """
    Simple single-head soft attention.
    """
    def __init__(self, input_dim, q_dim):
        super().__init__()
        self.q_dim = q_dim
        # nn.Linear is automatically initialised and includes bias already
        self.query_projection = nn.Linear(input_dim, q_dim)
        # Should have the same output dimensionality as the query projection
        self.key_projection = nn.Linear(input_dim, q_dim)
        self.value_projection = nn.Linear(input_dim, input_dim)

    def forward(self, X, causal_masking=True):
        """
        For simplicity we assume that we are training on a single sequence
        at a time, so the input shape is (sequence_length, input_dim).

        What will we need to do to handle padding tokens in the input?
        """
        # (sequence_length, q_dim)
        Q = self.query_projection(X)
        K = self.key_projection(X)
        # (sequence_length, input_dim)
        V = self.value_projection(X)
        # (sequence_length, sequence_length)
        # We need to transpose the keys for multiplication,
        # but we cannot use K.T because the first dimension is
        # the batch dimension, and we need to keep it, so
        # we specify dimensions explicitly.
        scores = Q @ K.transpose(-2, -1)

        # Normalisation
        scores = scores / math.sqrt(self.q_dim)

        # When doing autoregressive language modelling, we do not allow
        # tokens to attend to tokens on the right because this gives away the
        # answer. A simple way to do this is to ensure that the softmax scores
        # for tokens to the right of the target token are always zero. In order
        # to insure that, we can replace the corresponding attention scores with
        # negative infinity, so that exp(score) = 0.
        if causal_masking:
            scores += torch.triu(
                # full_like will automatically put the mask on the correct device
                torch.full_like(
                    scores,
                    float("-inf")
                ),
            diagonal=1)

        # How do we mask out PAD tokens?
        # We will have a padding mask with 1s for PAD tokens
        # of size batch_size x max_sequence_length
        # We will have a pad mask for each input
        # We add the masks

        attention_weights = torch.softmax(scores, dim=-1)
        return attention_weights @ V


class LearnedPositionalEmbedding(nn.Module):
    """
    Minimal learned positional embeddings (RoBERTa/BERT style).

    - Uses an nn.Embedding table for positions.
    - Supports padding via `padding_idx` (positions for pad tokens get 0 vector).
    - Adds position embeddings to token embeddings.
    """

    def __init__(self, d_model: int, max_len: int = 514, padding_idx: int = 1):
        """
        Args:
            d_model: hidden size
            max_len: max sequence length supported (often 512 + 2 specials)
            padding_idx: index in input_ids used for padding (RoBERTa typically 1)
        """
        super().__init__()
        self.max_len = max_len
        self.padding_idx = padding_idx
        # +1 because cumsum produces positions 1..max_len for non-pad tokens,
        # so we need indices 0..max_len (max_len + 1 rows total).
        self.position_embeddings = nn.Embedding(max_len + 1, d_model, padding_idx=0)

        # We'll map pad positions to index 0 in the position table.
        # (i.e., row 0 stays zeroed and is never updated)
        with torch.no_grad():
            self.position_embeddings.weight[0].zero_()

    def forward(self, x: torch.Tensor, input_ids: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: token embeddings, shape (batch, seq_len, d_model)
            input_ids: token ids, shape (batch, seq_len)

        Returns:
            x + learned position embeddings, same shape as x
        """
        _, seq_len, _ = x.shape
        if seq_len > self.max_len:
            raise ValueError(f"seq_len={seq_len} exceeds max_len={self.max_len}")

        # RoBERTa-style: positions start at padding_idx+1 for non-pad tokens,
        # and pad tokens get position id 0 (so embedding is zero).
        mask = (input_ids != self.padding_idx).to(torch.long)
        position_ids = torch.cumsum(mask, dim=1) * mask

        # self.position_embeddings is a module, not a tensor, so it needs to be called.
        pos_emb = self.position_embeddings(position_ids)  # (B, T, D)
        # Additive encoding
        return x + pos_emb


class LayerNorm(nn.Module):
    """
    BERT-style LayerNorm: normalise over last dimension with learnable
    scale (gamma) and bias (beta).
    """

    def __init__(self, hidden_size: int, eps: float = 1e-12):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(hidden_size))   # gamma
        self.bias = nn.Parameter(torch.zeros(hidden_size))    # beta

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return (x - x.mean(dim=-1, keepdim=True)) / torch.sqrt(
            torch.var(x, dim=-1, correction=0, keepdim=True)  + self.eps
        ) * self.weight + self.bias


class MLP(nn.Module):
    """
    Just an MLP, applied elementwise over the last dimension.
    """
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.fully_connected1 = nn.Linear(input_dim, hidden_dim)
        self.fully_connected2 = nn.Linear(hidden_dim, input_dim)
        self.relu = nn.ReLU()
        self.everything = nn.Sequential(
            self.fully_connected1,
            self.relu,
            self.fully_connected2
        )

    def forward(self, X):
        return self.everything(X)


class TransformerLayer(nn.Module):
    """
    Transformer layers do not change the shape of the input, so we can
    arbitrarily stack them.

    Transformer layer consist of sublayers and residual connection. A popular
    sequence looks like this:

    1. LayerNorm #1
    2. Attention
    3. Residual connection #1
    4. LayerNorm #2
    5. MLP
    6. Residual connection #2

    NB: don't forget add dropout
    """
    def __init__(self, input_dim, query_dim, mlp_hidden_dim, dropout_p=0.1):
        super().__init__()

        self.sublayer1 = nn.Sequential(
            LayerNorm(input_dim),
            # We're calling attention with the default causal masking.
            Attention(input_dim, query_dim),
            nn.Dropout(dropout_p)
        )

        self.sublayer2 = nn.Sequential(
            LayerNorm(input_dim),
            MLP(input_dim, mlp_hidden_dim),
            nn.Dropout(dropout_p)
        )

    def forward(self, X, causal_masking=True):
        if not causal_masking:
            raise NotImplementedError('This implementation assumes causal masking.')
        X = X + self.sublayer1(X)
        return X + self.sublayer2(X)


class LanguageModellingHead(nn.Module):
    """
    A simple linear layer to project the transformer output to the vocabulary size.
    """
    def __init__(self, input_dim, vocab_size):
        super().__init__()
        self.projection = nn.Linear(input_dim, vocab_size)

        # Manual version
        # self.weights = nn.Parameter(TENSOR torch.randn(input_dim, vocab_size))
        # self.bias = nn.Parameter(torch.zeros(vocab_size))

    def forward(self, x):
        # (..., input_dim) -> (..., vocab_size)
        return self.projection(x)


class NLayerTransformer(nn.Module):
    def __init__(
        self,
        n_layers,
        vocab_size,
        input_dim,
        max_len,
        padding_idx,
        query_dim,
        mlp_hidden_dim,
        dropout_p=0.1
    ):
        super().__init__()

        self.lexical_embedding = LexicalEmbedding(vocab_size, input_dim, padding_idx)
        self.positional_embedding = LearnedPositionalEmbedding(input_dim, max_len)
        # If we want to control masking, we can instantiate the layers separately and
        # call them individually in the forward method.
        # self.transformer_layer1 = TransformerLayer(input_dim, query_dim, mlp_hidden_dim, dropout_p)
        # etc.
        # Or (if we want default causal masking)
        self.layers = nn.Sequential(*[
            TransformerLayer(input_dim, query_dim, mlp_hidden_dim, dropout_p)
             for _ in range(n_layers)
        ])
        self.lm_head = LanguageModellingHead(input_dim, vocab_size)
        self.padding_idx = padding_idx
        self.vocab_size = vocab_size

    def forward(self, input_ids, causal_masking=True):
        result = {
            'logits': None,
            'last_hidden_state': None
        }
        lexical_embeddings = self.lexical_embedding(input_ids)
        combined_embeddings = self.positional_embedding(lexical_embeddings, input_ids)
        last_hidden_state = self.layers(combined_embeddings)
        result['last_hidden_state'] = last_hidden_state
        logits = self.lm_head(last_hidden_state)
        result['logits'] = logits
        return result


## Training code

In [ ]:
def compute_lm_loss(logits, input_ids, vocab_size, loss_fn):
    # Create target labels by shifting input_ids
    # We will have one less token in the target than in the inputs
    # because we cannot predict the first token in each sequence.
    target_labels = input_ids.clone()

    # Shift the targets
    # Input: 	   This is  the    [input.]
    # Gold labels: is   the input.
    target_labels = target_labels[:, 1:]
    logits = logits[:, :-1]

    # Turn the logits 3d batch tensor into a 2d matrix of shape
    # (batch_size * seq_len, vocab_size) to compute the loss for each row
    logits = logits.reshape(-1, vocab_size)

    # Change the shape of the labels to be compatible with this input
    target_labels = target_labels.flatten()

    return loss_fn(logits, target_labels)


def train_lm_step(model, input_ids, optimizer, vocab_size, loss_fn):
    model.train()
    # TODO


def train_lm_epoch(model, dataloader, optimizer, vocab_size, loss_fn, no_progress_bar=False):
    # TODO


def validate_lm_epoch(model, dataloader, vocab_size, loss_fn, pad_token_id=0, no_progress_bar=False):
    """
    Some things we may do here:
    1. Compute the cross-entropy loss on the validation tokens. (This will also give us perplexity, which is
       the total loss for the batch divided by the number of non-pad tokens in it.)
    2. Compute the prediction accuracy (proportion of next-tokens that were predicted correctly given the prefix).
       We may also want to return these tokens to inspect what next tokens are selected for different prefixes.
    """
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_tokens = 0
    all_generated = []
    with torch.no_grad():
        # TODO

    return {
        "loss": None,
        "accuracy": None,
        "generated_sequences": None,
        "perplexity": None
    }


## Tokenisation code

In [ ]:
def byte_tokenise(inputs):
    if type(inputs) == str:
        # Or: if isinstance(inputs, str):
        inputs = [inputs]

    input_bytes = list(map(
        lambda s: list(map(int, s.encode())),
        inputs
    ))

    # Add padding to make all inputs the same length.
    max_len = max(map(len, input_bytes))
    result = torch.zeros((len(input_bytes), max_len), dtype=torch.long)
    for i, input_bytes_i in enumerate(input_bytes):
        result[i, :len(input_bytes_i)] = torch.tensor(input_bytes_i, dtype=torch.long)

    return result


def decode_seq(seq, special_tokens: set):
    byte_list = [b for b in seq.tolist() if b not in special_tokens]
    return bytes(byte_list).decode("utf-8", errors="replace")

## Generation code

In [ ]:
def greedy_generate(
    model: NLayerTransformer,
    prefix: torch.Tensor,
    max_len: int,
    eos_idx: int,
    pad_idx: int,
) -> torch.Tensor:
    """
    Greedily generate tokens from a prefix until max_len is reached or
    eos_idx / pad_idx is produced. Return the complete sequence.
    """
    model.eval()
    # Work with a (1, T) batch.
    ids = prefix.unsqueeze(0)  # (1, prefix_len)

    with torch.no_grad():
        pass
        # TODO

    return ids.squeeze(0)  # (T,)

## The actual training

In [ ]:
# Obtaining the data

!wget "https://gist.githubusercontent.com/macleginn/46b41588d48c5c8f7190cb03a3cab0b2/raw/bdd2969c34e517d24d2d169be3b32889484bc972/gistfile1.txt"
!mv gistfile1.txt mfw_3000.txt

In [ ]:
# Normally these things are provided as command-line arguments using the argparse library,
# but for simplicity we will just hard-code them here.

# These two parameters can have a serious impact on the end result!
BATCH_SIZE = 32
LEARNING_RATE = 1e-4

MAX_EPOCHS = 100
PATIENCE = 5
PATH_TO_DATA = "mfw_3000.txt"
N_LAYERS = 3
VOCAB_SIZE = 256
INPUT_DIM = 64
# Maximum length in the data + sos + eos.
MAX_LEN = 18
PADDING_IDX = 0

# Here we assume that these two tokens will never be
# found in the input. What should we do if we don't have
# such a guarantee?
SOS_TOKEN = '^'  # Start-of-sequence token
EOS_TOKEN = '$'  # End-of-sequence token
SOS_IDX = int(SOS_TOKEN.encode()[0])
EOS_IDX = int(EOS_TOKEN.encode()[0])

QUERY_DIM = 32
MLP_HIDDEN_DIM = 128
DROPOUT_P = 0.1

# Set random seed
# NB: set the seed separately for numpy and the built-in random module if you use them.
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

with open(PATH_TO_DATA, "r", encoding="utf-8") as f:
    # Add start-of-sequence and end-of-sequence tokens, so that the model can learn to predict when sequences end.
    words = [f'{SOS_TOKEN}{line.strip()}{EOS_TOKEN}' for line in f]

input_ids = byte_tokenise(words).to(device)
print(f"input_ids.shape: {input_ids.shape}")
# Randomly split into, train, validation, and test sets.
input_idx = torch.randperm(len(input_ids))
train_idx = input_idx[: int(0.8 * len(input_ids))].to(device)
val_idx = input_idx[int(0.8 * len(input_ids)) : int(0.9 * len(input_ids))].to(device)
test_idx = input_idx[int(0.9 * len(input_ids)) :].to(device)
print(f"train: {len(train_idx)}, val: {len(val_idx)}, test: {len(test_idx)}")

train_dataset = TensorDataset(input_ids[train_idx])
val_dataset = TensorDataset(input_ids[val_idx])
test_dataset = TensorDataset(input_ids[test_idx])

# The data are preshuffled, so shuffle=True is not really necessary.
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

model = NLayerTransformer(
    n_layers=N_LAYERS,
    vocab_size=VOCAB_SIZE,
    input_dim=INPUT_DIM,
    max_len=MAX_LEN,
    padding_idx=PADDING_IDX,
    query_dim=QUERY_DIM,
    mlp_hidden_dim=MLP_HIDDEN_DIM,
    dropout_p=DROPOUT_P,
)
model.to(device)

loss_fn = CrossEntropyLoss(ignore_index=0)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
best_val_accuracy = float("-inf")
# Train the model for a maximum of MAX_EPOCHS epochs,
# but stop early if the validation accuracy doesn't improve for PATIENCE epochs.
# NB: accuracy may continue to improve even if the loss doesn't.
# Whenever you can, always use the actual target metric to guide the model.

for epoch in range(MAX_EPOCHS):
    train_loss = train_lm_epoch(model, train_dataloader, optimizer, VOCAB_SIZE, loss_fn)
    val_metrics = validate_lm_epoch(
        model, val_dataloader, VOCAB_SIZE, loss_fn, pad_token_id=PADDING_IDX
    )

    print(
        f"Epoch {epoch+1}/{MAX_EPOCHS} - Train Loss: {train_loss:.4f} - Val Loss: "
        f"{val_metrics['loss']:.4f} - Val Accuracy: {val_metrics['accuracy']:.4f}"
    )

    if val_metrics["accuracy"] > best_val_accuracy:
        best_val_accuracy = val_metrics["accuracy"]
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print("Early stopping triggered.")
        break

test_metrics = validate_lm_epoch(
    model, test_dataloader, VOCAB_SIZE, loss_fn, pad_token_id=PADDING_IDX
)
print(
    f"Test Loss: {test_metrics['loss']:.4f} - Test Accuracy: {test_metrics['accuracy']:.4f}"
)

SPECIAL_TOKENS = {PADDING_IDX, SOS_IDX, EOS_IDX}

print("\nFirst 10 generated vs. gold sequences from the test set:")
gold_sequences = input_ids[test_idx].cpu()
for gold, generated in zip(gold_sequences[:10], test_metrics["generated_sequences"][:10]):
    # Start generating from the first two characters of the gold sequence.
    prefix = gold[:3].to(device)
    greedy = greedy_generate(model, prefix, MAX_LEN, EOS_IDX, PADDING_IDX).cpu()
    print(
        f"  gold: {decode_seq(gold, SPECIAL_TOKENS)!r:20s}"
        f"  teacher-forced: {decode_seq(generated, SPECIAL_TOKENS)!r:20s}"
        f"  greedy: {decode_seq(greedy, SPECIAL_TOKENS)!r}"
    )
